In [1]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      12
  On-line CPU(s) list:       0-11
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   85
    Thread(s) per core:      2
    Core(s) per socket:      6
    Socket(s):               1
    Stepping:                7
    BogoMIPS:                4400.30
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                 

In [2]:
!nvidia-smi

Sun Apr 26 15:24:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")

PyTorch: 2.10.0+cu128
CUDA available: True
cuDNN version: 91002


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os
from pathlib import Path

PROJECT_DIR = Path(os.environ.get(
    "AUV_PROJECT_DIR",
    "/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7",
))
assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"
%cd $PROJECT_DIR

/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7


In [7]:
%pip install -q torchdiffeq

# Phase-1A OC v4-lite formal workflow

**定位**：当前 Phase-1A 的正式 Colab 执行入口。

**目标**：干净执行 OC-only 三模型五 seed protocol-sensitivity check，判断 trajectory-consistent `v4_lite` noisy IC 是否改变当前 `iid_noisy_ic` 下的 PHNODE / structured dynamics 结论。

**边界**：

- 只跑 `oc + known-current surrogate`。
- 不复用旧 checkpoint / suite / proxy。
- 不生成旧 `phase1_*` 兼容产物。
- 不默认运行 Phase-1B 模型，也不默认运行 `degraded_eval` / `heading_biased_eval`。
- 每个长任务 cell 直接调用 `scripts/run_phase1a_oc_v4lite.sh`，保留实时 stdout/stderr。

## 0. Shared configuration

默认输出：

- training suites: `checkpoints/sweep_oc_phase1a_{smoke1,smoke3,decision}_{clean,iid,v4lite}_${RUN_TAG}`
- local proxy suites: `/content/_proxy_suites/sweep_oc_phase1a_{smoke1,smoke3,decision}_proxy_${RUN_TAG}`
- exported decision artifacts: `checkpoints/sweep_oc_phase1a_decision_proxy_${RUN_TAG}/phase1a_*`
- execution logs: `checkpoints/phase1a_logs/${RUN_TAG}/`
- run metadata: `checkpoints/phase1a_metadata_${RUN_TAG}/phase1a_{run_config,environment}.json`

如果需要重跑同一文档，先换 `RUN_TAG`。`preflight` 会拒绝覆盖已存在的目标目录。

In [8]:
import os

# Runtime
os.environ["PYTHON_BIN"] = "python"
os.environ["DEVICE"] = "cuda"
os.environ["LOCAL_PROXY_ROOT"] = "/content/_proxy_suites"

# Phase-1A identity
os.environ["RUN_TAG"] = "phase1a_oc_v4lite_cleanrun_v1"
os.environ["DATASET"] = "data/auv_oc_traj1000_blk150_s23_d0be9434.pkl"
os.environ["NOISE_REFERENCE"] = "remus100_dr"
os.environ["PHASE1A_LOG_DIR"] = str(PROJECT_DIR / "checkpoints" / "phase1a_logs" / os.environ["RUN_TAG"])
os.environ["PHASE1A_METADATA_DIR"] = str(PROJECT_DIR / "checkpoints" / f"phase1a_metadata_{os.environ['RUN_TAG']}")

# Phase-1A matrix
os.environ["PHASE1A_MODELS"] = "phnode_full ablate_no_mass_prior ablate_no_lift"
os.environ["SMOKE1_MODELS"] = "phnode_full"
os.environ["SMOKE_SEEDS"] = "42 44 46"
os.environ["DECISION_SEEDS"] = "42 43 44 45 46"

# Evaluation contract
os.environ["SMOKE_EVAL_NUM_TRAJ_PER_SCENARIO"] = "6"
os.environ["DECISION_EVAL_NUM_TRAJ_PER_SCENARIO"] = "30"
os.environ["EVAL_TIMES"] = "10 30 60"
os.environ["EVAL_SCENARIOS"] = "PRBS CHIRP OU"
os.environ["EVAL_BASE_SEED"] = "42"
os.environ["EVAL_NOISE_SEED"] = "2024"
os.environ["EVAL_PROGRESS_EVERY"] = "5"
os.environ["EVAL_NUM_DIAGNOSTIC_PLOTS"] = "6"
os.environ["IID_EVAL_PROFILES"] = "clean nominal_eval"
os.environ["V4_EVAL_PROFILES"] = "nominal_eval"

# Audit gate
os.environ["STRICT_ZERO_NOISE_AUDIT"] = "1"
os.environ["SOFT_MIN_EPOCH_SCALE"] = "0.05"

print("RUN_TAG=", os.environ["RUN_TAG"])
print("PHASE1A_MODELS=", os.environ["PHASE1A_MODELS"])
print("DECISION_SEEDS=", os.environ["DECISION_SEEDS"])
print("PHASE1A_LOG_DIR=", os.environ["PHASE1A_LOG_DIR"])
print("PHASE1A_METADATA_DIR=", os.environ["PHASE1A_METADATA_DIR"])

RUN_TAG= phase1a_oc_v4lite_cleanrun_v1
PHASE1A_MODELS= phnode_full ablate_no_mass_prior ablate_no_lift
DECISION_SEEDS= 42 43 44 45 46
PHASE1A_LOG_DIR= /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_logs/phase1a_oc_v4lite_cleanrun_v1
PHASE1A_METADATA_DIR= /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_metadata_phase1a_oc_v4lite_cleanrun_v1


## 1. Preflight

确认本次 clean run 的所有目标 suite / proxy 目录都不存在，并保存本次 run config / environment metadata。若这里失败，换 `RUN_TAG` 或手动清理目标目录。

In [ ]:
os.environ["MODE"] = "preflight"
!bash scripts/run_phase1a_oc_v4lite.sh

[phase1a]
MODE=preflight
RUN_TAG=phase1a_oc_v4lite_cleanrun_v1
DATASET=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
DEVICE=cuda
PHASE1A_MODELS=phnode_full ablate_no_mass_prior ablate_no_lift
SMOKE1_MODELS=phnode_full
SMOKE_SEEDS=42 44 46
DECISION_SEEDS=42 43 44 45 46
NOISE_REFERENCE=remus100_dr
PHASE1A_LOG_DIR=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_logs/phase1a_oc_v4lite_cleanrun_v1
PHASE1A_METADATA_DIR=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_metadata_phase1a_oc_v4lite_cleanrun_v1
Preflight passed: all Phase-1A target suite/proxy directories are absent.
[metadata] /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_metadata_phase1a_oc_v4lite_cleanrun_v1


## 2. Smoke-1 train + protocol validation

最小单模型 gate：`phnode_full × seeds 42/44/46 × clean/iid/v4lite`。

本 cell 完成训练、训练审计和 mandatory `v4_lite` protocol validation。

In [ ]:
# v4_lite 训练时间过长，进行暂停
os.environ["MODE"] = "smoke1_train"
!bash scripts/run_phase1a_oc_v4lite.sh

[phase1a]
MODE=smoke1_train
RUN_TAG=phase1a_oc_v4lite_cleanrun_v1
DATASET=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
DEVICE=cuda
PHASE1A_MODELS=phnode_full ablate_no_mass_prior ablate_no_lift
SMOKE1_MODELS=phnode_full
SMOKE_SEEDS=42 44 46
DECISION_SEEDS=42 43 44 45 46
NOISE_REFERENCE=remus100_dr
PHASE1A_LOG_DIR=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_logs/phase1a_oc_v4lite_cleanrun_v1
PHASE1A_METADATA_DIR=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_metadata_phase1a_oc_v4lite_cleanrun_v1
[train] suite=sweep_oc_phase1a_smoke1_clean_phase1a_oc_v4lite_cleanrun_v1 models=phnode_full seeds=42 44 46 protocol=auto
Training all models with profile-based noisy IC.
Dataset: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
Group: all
Explicit models: phnode_full
Seeds: 42 44 46
Noise config: protocol=auto, profile=clea

In [ ]:
# 第一次修改 140s -> 14s
os.environ["MODE"] = "smoke1_train_v4lite_resume"
!bash scripts/run_phase1a_oc_v4lite.sh

[phase1a]
MODE=smoke1_train_v4lite_resume
RUN_TAG=phase1a_oc_v4lite_cleanrun_v1
DATASET=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
DEVICE=cuda
PHASE1A_MODELS=phnode_full ablate_no_mass_prior ablate_no_lift
SMOKE1_MODELS=phnode_full
SMOKE_SEEDS=42 44 46
DECISION_SEEDS=42 43 44 45 46
NOISE_REFERENCE=remus100_dr
PHASE1A_LOG_DIR=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_logs/phase1a_oc_v4lite_cleanrun_v1
PHASE1A_METADATA_DIR=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_metadata_phase1a_oc_v4lite_cleanrun_v1
[resume-train] suite=sweep_oc_phase1a_smoke1_v4lite_phase1a_oc_v4lite_cleanrun_v1 models=phnode_full seeds=42 44 46 protocol=v4_lite
Training all models with profile-based noisy IC.
Dataset: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
Group: all
Explicit models: phnode_full
Seeds: 42 44 46
Noise config: pr

In [ ]:
# 第二次修改 14s -> 6s
os.environ["MODE"] = "smoke1_train_v4lite_resume"
!bash scripts/run_phase1a_oc_v4lite.sh

[phase1a]
MODE=smoke1_train_v4lite_resume
RUN_TAG=phase1a_oc_v4lite_cleanrun_v1
DATASET=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
DEVICE=cuda
PHASE1A_MODELS=phnode_full ablate_no_mass_prior ablate_no_lift
SMOKE1_MODELS=phnode_full
SMOKE_SEEDS=42 44 46
DECISION_SEEDS=42 43 44 45 46
NOISE_REFERENCE=remus100_dr
PHASE1A_LOG_DIR=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_logs/phase1a_oc_v4lite_cleanrun_v1
PHASE1A_METADATA_DIR=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_metadata_phase1a_oc_v4lite_cleanrun_v1
[resume-train] suite=sweep_oc_phase1a_smoke1_v4lite_phase1a_oc_v4lite_cleanrun_v1 models=phnode_full seeds=42 44 46 protocol=v4_lite
Training all models with profile-based noisy IC.
Dataset: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
Group: all
Explicit models: phnode_full
Seeds: 42 44 46
Noise config: pr

## 3. Smoke-1 rollout eval + local proxy

使用小 rollout 数检查 eval、summary 和 report 路径。Smoke 只验证流程，不写研究结论。

In [ ]:
os.environ["MODE"] = "smoke1_eval"
!bash scripts/run_phase1a_oc_v4lite.sh

[phase1a]
MODE=smoke1_eval
RUN_TAG=phase1a_oc_v4lite_cleanrun_v1
DATASET=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
DEVICE=cuda
PHASE1A_MODELS=phnode_full ablate_no_mass_prior ablate_no_lift
SMOKE1_MODELS=phnode_full
SMOKE_SEEDS=42 44 46
DECISION_SEEDS=42 43 44 45 46
NOISE_REFERENCE=remus100_dr
PHASE1A_LOG_DIR=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_logs/phase1a_oc_v4lite_cleanrun_v1
PHASE1A_METADATA_DIR=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_metadata_phase1a_oc_v4lite_cleanrun_v1
[eval] suite=sweep_oc_phase1a_smoke1_clean_phase1a_oc_v4lite_cleanrun_v1 eval_protocol=iid_noisy_ic profiles=clean nominal_eval
Suite directory: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_smoke1_clean_phase1a_oc_v4lite_cleanrun_v1
Manifest: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_smoke1_cle

## 4. Smoke-3 train + protocol validation

三模型 smoke：`phnode_full / ablate_no_mass_prior / ablate_no_lift × seeds 42/44/46`。

In [ ]:
os.environ["MODE"] = "smoke3_train"
!bash scripts/run_phase1a_oc_v4lite.sh

Streaming output truncated to the last 5000 lines.
[nominal_eval]
position_rmse: ratio=10.266  degradation=+926.6%
rotation_geodesic: ratio=17.602  degradation=+1660.2%
velocity_rmse: ratio=3.404  degradation=+240.4%
angular_rmse: ratio=2.750  degradation=+175.0%
success_rate: ratio=1.000  degradation=+0.0%
trajectory_failure_rate: ratio=1.000  degradation=+0.0%
[degraded_eval]
position_rmse: ratio=24.171  degradation=+2317.1%
rotation_geodesic: ratio=43.027  degradation=+4202.7%
velocity_rmse: ratio=6.730  degradation=+573.0%
angular_rmse: ratio=5.116  degradation=+411.6%
success_rate: ratio=1.000  degradation=+0.0%
trajectory_failure_rate: ratio=1.000  degradation=+0.0%
[heading_biased_eval]
position_rmse: ratio=37.091  degradation=+3609.1%
rotation_geodesic: ratio=71.601  degradation=+7060.1%
velocity_rmse: ratio=5.445  degradation=+444.5%
angular_rmse: ratio=2.750  degradation=+175.0%
success_rate: ratio=1.000  degradation=+0.0%
trajectory_failure_rate: ratio=1.000  degradation=+0.

## 5. Smoke-3 rollout eval + local proxy

通过后再进入五 seed decision run。

In [ ]:
os.environ["MODE"] = "smoke3_eval"
!bash scripts/run_phase1a_oc_v4lite.sh

Streaming output truncated to the last 5000 lines.

Diagnostic Cases To Max Horizon
--------------------------------------------------------------------------------
Largest completed terminal errors
  PRBS seed=46 | reason=completed | sim_time=60.00s | pos=3.3922 m | rot=0.0506 rad | total_vel_violation=0
  OU seed=200042 | reason=completed | sim_time=60.00s | pos=2.6853 m | rot=0.0661 rad | total_vel_violation=0
  OU seed=200043 | reason=completed | sim_time=60.00s | pos=2.0107 m | rot=0.0698 rad | total_vel_violation=0
  PRBS seed=42 | reason=completed | sim_time=60.00s | pos=1.8868 m | rot=0.0253 rad | total_vel_violation=0
  CHIRP seed=100047 | reason=completed | sim_time=60.00s | pos=1.6587 m | rot=0.0337 rad | total_vel_violation=0

Diagnostic Plot Files
--------------------------------------------------------------------------------
PRBS seed=46 | label=high_terminal_error | /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_smoke3_iid_phase1a_

In [ ]:
import shutil
from pathlib import Path

run_tag = "phase1a_oc_v4lite_cleanrun_v1"
proxy = Path("/content/_proxy_suites") / f"sweep_oc_phase1a_smoke3_proxy_{run_tag}"

if proxy.exists():
    shutil.rmtree(proxy)
    print("removed partial proxy:", proxy)
else:
    print("no partial proxy found")


no partial proxy found


In [ ]:
import os
os.environ["MODE"] = "smoke3_eval"
!bash scripts/run_phase1a_oc_v4lite.sh


[phase1a]
MODE=smoke3_eval
RUN_TAG=phase1a_oc_v4lite_cleanrun_v1
DATASET=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
DEVICE=cuda
PHASE1A_MODELS=phnode_full ablate_no_mass_prior ablate_no_lift
SMOKE1_MODELS=phnode_full
SMOKE_SEEDS=42 44 46
DECISION_SEEDS=42 43 44 45 46
NOISE_REFERENCE=remus100_dr
PHASE1A_LOG_DIR=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_logs/phase1a_oc_v4lite_cleanrun_v1
PHASE1A_METADATA_DIR=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_metadata_phase1a_oc_v4lite_cleanrun_v1
[eval] suite=sweep_oc_phase1a_smoke3_clean_phase1a_oc_v4lite_cleanrun_v1 eval_protocol=iid_noisy_ic profiles=clean nominal_eval
Suite directory: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_smoke3_clean_phase1a_oc_v4lite_cleanrun_v1
Manifest: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_smoke3_cle

## 6. Decision train + protocol validation

正式 Phase-1A decision matrix：三模型 × seeds `42/43/44/45/46` × `clean/iid_noisy_ic/v4_lite`。

In [ ]:
os.environ["MODE"] = "decision_train"
!bash scripts/run_phase1a_oc_v4lite.sh

[phase1a]
MODE=decision_train
RUN_TAG=phase1a_oc_v4lite_cleanrun_v1
DATASET=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
DEVICE=cuda
PHASE1A_MODELS=phnode_full ablate_no_mass_prior ablate_no_lift
SMOKE1_MODELS=phnode_full
SMOKE_SEEDS=42 44 46
DECISION_SEEDS=42 43 44 45 46
NOISE_REFERENCE=remus100_dr
PHASE1A_LOG_DIR=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_logs/phase1a_oc_v4lite_cleanrun_v1
PHASE1A_METADATA_DIR=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/phase1a_metadata_phase1a_oc_v4lite_cleanrun_v1
[train] suite=sweep_oc_phase1a_decision_clean_phase1a_oc_v4lite_cleanrun_v1 models=phnode_full ablate_no_mass_prior ablate_no_lift seeds=42 43 44 45 46 protocol=auto
Training all models with profile-based noisy IC.
Dataset: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
Group: all
Explicit models: phnode_full ablate_no_m

In [ ]:
# 只补训 43/45

!export RUN_TAG=phase1a_oc_v4lite_cleanrun_v1
!export DATASET=data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
!export DEVICE=cuda
!export NOISE_REFERENCE=remus100_dr
!export PHASE1A_MODELS="phnode_full ablate_no_mass_prior ablate_no_lift"

!bash scripts/train_all_models_noise_profile.sh \
  --profile oc \
  --models "${PHASE1A_MODELS}" \
  --dataset "${DATASET}" \
  --seeds "43 45" \
  --suite-name "sweep_oc_phase1a_decision_extra43-45_clean_${RUN_TAG}" \
  --noise-profile clean \
  --noise-protocol auto \
  --noise-reference "${NOISE_REFERENCE}" \
  --block-eval-noise-profiles clean \
  --heldout-eval-noise-profiles clean \
  --device "${DEVICE}"

!bash scripts/train_all_models_noise_profile.sh \
  --profile oc \
  --models "${PHASE1A_MODELS}" \
  --dataset "${DATASET}" \
  --seeds "43 45" \
  --suite-name "sweep_oc_phase1a_decision_extra43-45_iid_${RUN_TAG}" \
  --noise-profile nominal_train \
  --noise-protocol iid_noisy_ic \
  --noise-reference "${NOISE_REFERENCE}" \
  --block-eval-noise-profiles clean \
  --heldout-eval-noise-profiles clean \
  --device "${DEVICE}"

!bash scripts/train_all_models_noise_profile.sh \
  --profile oc \
  --models "${PHASE1A_MODELS}" \
  --dataset "${DATASET}" \
  --seeds "43 45" \
  --suite-name "sweep_oc_phase1a_decision_extra43-45_v4lite_${RUN_TAG}" \
  --noise-profile nominal_train \
  --noise-protocol v4_lite \
  --noise-reference "${NOISE_REFERENCE}" \
  --block-eval-noise-profiles clean \
  --heldout-eval-noise-profiles clean \
  --device "${DEVICE}"


Training all models with profile-based noisy IC.
Dataset: data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
Group: all
Explicit models: phnode_full ablate_no_mass_prior ablate_no_lift
Seeds: 43 45
Noise config: protocol=auto, profile=clean, scale=1.0, warmup=20, ramp=80, mix_ratio=0.5
Auto-eval profiles: block=[clean] heldout=[clean]
Suite directory: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_extra43-45_clean_phase1a_oc_v4lite_cleanrun_v1
Dataset: data/auv_oc_traj1000_blk150_s23_d0be9434.pkl
Models (3): phnode_full ablate_no_mass_prior ablate_no_lift
Seeds: 43 45
[skip] main_phnode_full_seed43 already exists at /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_extra43-45_clean_phase1a_oc_v4lite_cleanrun_v1/main_phnode_full_seed43/best_model.pt
[skip] main_phnode_full_seed45 already exists at /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_extra43-45

In [ ]:
# 注册五 seed decision suite

from pathlib import Path
import os
import pandas as pd

run_tag = os.environ.get("RUN_TAG", "phase1a_oc_v4lite_cleanrun_v1")
root = Path.cwd()
ckpt = root / "checkpoints"

models = {"phnode_full", "ablate_no_mass_prior", "ablate_no_lift"}
smoke_seeds = {42, 44, 46}
extra_seeds = {43, 45}
decision_seeds = {42, 43, 44, 45, 46}

def suite_path(phase, protocol):
    path = ckpt / f"sweep_oc_phase1a_{phase}_{protocol}_{run_tag}"
    if not path.is_dir():
        raise FileNotFoundError(f"Missing suite: {path}")
    if not (path / "runs.tsv").is_file():
        raise FileNotFoundError(f"Missing manifest: {path / 'runs.tsv'}")
    return path

def read_runs(suite_dir):
    runs = pd.read_csv(suite_dir / "runs.tsv", sep="\t")
    runs["seed"] = runs["seed"].astype(int)
    return runs

def validate_suite(suite_dir, expected_seeds):
    runs = read_runs(suite_dir)
    got_models = set(runs["model_type"])
    got_seeds = set(runs["seed"])
    if got_models != models:
        raise ValueError(f"{suite_dir.name}: model mismatch: {got_models}")
    if got_seeds != expected_seeds:
        raise ValueError(f"{suite_dir.name}: seed mismatch: {got_seeds}")
    expected_n = len(models) * len(expected_seeds)
    if len(runs) != expected_n:
        raise ValueError(f"{suite_dir.name}: expected {expected_n} runs, got {len(runs)}")
    missing_ckpt = [p for p in runs["checkpoint"] if not Path(p).is_file()]
    if missing_ckpt:
        raise FileNotFoundError(f"{suite_dir.name}: missing checkpoints: {missing_ckpt[:3]}")
    print(f"[ok] {suite_dir.name}: {len(runs)} runs")
    return runs

for protocol in ["clean", "iid", "v4lite"]:
    validate_suite(suite_path("smoke3", protocol), smoke_seeds)

for protocol in ["clean", "iid", "v4lite"]:
    validate_suite(suite_path("decision_extra43-45", protocol), extra_seeds)

for protocol in ["clean", "iid", "v4lite"]:
    smoke_suite = suite_path("smoke3", protocol)
    extra_suite = suite_path("decision_extra43-45", protocol)
    target_suite = ckpt / f"sweep_oc_phase1a_decision_{protocol}_{run_tag}"

    if target_suite.exists():
        raise FileExistsError(f"Refusing to overwrite existing suite: {target_suite}")

    rows = pd.concat([
        read_runs(smoke_suite).query("seed in @smoke_seeds"),
        read_runs(extra_suite).query("seed in @extra_seeds"),
    ], ignore_index=True)

    if set(rows["seed"]) != decision_seeds:
        raise ValueError(f"{protocol}: decision seed mismatch: {set(rows['seed'])}")

    target_suite.mkdir(parents=True)
    out_rows = []

    for _, row in rows.iterrows():
        source_run_dir = Path(row["run_dir"]).resolve()
        proxy_run_dir = target_suite / row["run_name"]
        proxy_run_dir.mkdir(parents=True)

        for child in source_run_dir.iterdir():
            if child.name == "rollout_benchmark":
                continue
            link = proxy_run_dir / child.name
            link.symlink_to(child.resolve(), target_is_directory=child.is_dir())

        out_rows.append({
            "group": row["group"],
            "model_type": row["model_type"],
            "seed": int(row["seed"]),
            "run_name": row["run_name"],
            "run_dir": str(proxy_run_dir),
            "checkpoint": str(proxy_run_dir / "best_model.pt"),
        })

    out = pd.DataFrame(out_rows).sort_values(["group", "model_type", "seed"])
    out.to_csv(target_suite / "runs.tsv", sep="\t", index=False)
    (target_suite / "suite_config.txt").write_text(
        f"Suite: {target_suite.name}\n"
        f"Type: decision proxy from smoke3 seeds 42/44/46 + extra seeds 43/45\n"
        f"Protocol: {protocol}\n",
        encoding="utf-8",
    )
    print(f"[registered] {target_suite.name}: {len(out)} runs")


[ok] sweep_oc_phase1a_smoke3_clean_phase1a_oc_v4lite_cleanrun_v1: 9 runs
[ok] sweep_oc_phase1a_smoke3_iid_phase1a_oc_v4lite_cleanrun_v1: 9 runs
[ok] sweep_oc_phase1a_smoke3_v4lite_phase1a_oc_v4lite_cleanrun_v1: 9 runs
[ok] sweep_oc_phase1a_decision_extra43-45_clean_phase1a_oc_v4lite_cleanrun_v1: 6 runs
[ok] sweep_oc_phase1a_decision_extra43-45_iid_phase1a_oc_v4lite_cleanrun_v1: 6 runs
[ok] sweep_oc_phase1a_decision_extra43-45_v4lite_phase1a_oc_v4lite_cleanrun_v1: 6 runs


OSError: [Errno 95] Operation not supported: '/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_smoke3_clean_phase1a_oc_v4lite_cleanrun_v1/main_phnode_full_seed42/training.log' -> '/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_clean_phase1a_oc_v4lite_cleanrun_v1/main_phnode_full_seed42/training.log'

In [ ]:
from pathlib import Path
import pandas as pd

run_tag = "phase1a_oc_v4lite_cleanrun_v1"
ckpt = Path.cwd() / "checkpoints"

models = {"phnode_full", "ablate_no_mass_prior", "ablate_no_lift"}
decision_seeds = {42, 43, 44, 45, 46}

def inspect_decision_suite(protocol):
    suite = ckpt / f"sweep_oc_phase1a_decision_{protocol}_{run_tag}"
    if not suite.exists():
        return protocol, "missing", suite

    manifest = suite / "runs.tsv"
    if not manifest.exists():
        return protocol, "partial_missing_runs_tsv", suite

    runs = pd.read_csv(manifest, sep="\t")
    runs["seed"] = runs["seed"].astype(int)

    bad = []
    if set(runs["model_type"]) != models:
        bad.append(f"models={sorted(set(runs['model_type']))}")
    if set(runs["seed"]) != decision_seeds:
        bad.append(f"seeds={sorted(set(runs['seed']))}")
    if len(runs) != 15:
        bad.append(f"n_runs={len(runs)}")

    missing_ckpt = [p for p in runs["checkpoint"] if not Path(p).is_file()]
    if missing_ckpt:
        bad.append(f"missing_ckpt={len(missing_ckpt)}")

    if bad:
        return protocol, "partial_or_invalid: " + "; ".join(bad), suite
    return protocol, "ok", suite

statuses = [inspect_decision_suite(p) for p in ["clean", "iid", "v4lite"]]
for protocol, status, suite in statuses:
    print(f"[{status}] {protocol}: {suite}")

if all(status == "ok" for _, status, _ in statuses):
    print("\nAll decision suites are complete. Skip registration and continue to audit/eval.")
else:
    print("\nSome decision suites are missing or partial. Remove only the partial decision proxy suites, then rerun registration.")


[partial_or_invalid: models=[]; seeds=[]; n_runs=0] clean: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_clean_phase1a_oc_v4lite_cleanrun_v1
[missing] iid: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_iid_phase1a_oc_v4lite_cleanrun_v1
[missing] v4lite: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_v4lite_phase1a_oc_v4lite_cleanrun_v1

Some decision suites are missing or partial. Remove only the partial decision proxy suites, then rerun registration.


In [ ]:
import shutil
from pathlib import Path

run_tag = "phase1a_oc_v4lite_cleanrun_v1"
ckpt = Path.cwd() / "checkpoints"

# Only set protocols listed as partial_or_invalid or partial_missing_runs_tsv.
protocols_to_remove = ["clean"]  # edit this list based on the inspection output

for protocol in protocols_to_remove:
    suite = ckpt / f"sweep_oc_phase1a_decision_{protocol}_{run_tag}"
    if suite.exists():
        shutil.rmtree(suite)
        print("removed:", suite)


removed: /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_clean_phase1a_oc_v4lite_cleanrun_v1


In [ ]:
from pathlib import Path
import shutil
import pandas as pd

run_tag = "phase1a_oc_v4lite_cleanrun_v1"
ckpt = Path.cwd() / "checkpoints"

models = {"phnode_full", "ablate_no_mass_prior", "ablate_no_lift"}
smoke_seeds = {42, 44, 46}
extra_seeds = {43, 45}
decision_seeds = {42, 43, 44, 45, 46}

def suite_path(phase, protocol):
    path = ckpt / f"sweep_oc_phase1a_{phase}_{protocol}_{run_tag}"
    if not path.is_dir():
        raise FileNotFoundError(f"Missing suite: {path}")
    if not (path / "runs.tsv").is_file():
        raise FileNotFoundError(f"Missing manifest: {path / 'runs.tsv'}")
    return path

def read_runs(suite_dir):
    runs = pd.read_csv(suite_dir / "runs.tsv", sep="\t")
    runs["seed"] = runs["seed"].astype(int)
    return runs

def validate_suite(suite_dir, expected_seeds):
    runs = read_runs(suite_dir)
    got_models = set(runs["model_type"])
    got_seeds = set(runs["seed"])

    if got_models != models:
        raise ValueError(f"{suite_dir.name}: model mismatch: {got_models}")
    if got_seeds != expected_seeds:
        raise ValueError(f"{suite_dir.name}: seed mismatch: {got_seeds}")

    expected_n = len(models) * len(expected_seeds)
    if len(runs) != expected_n:
        raise ValueError(f"{suite_dir.name}: expected {expected_n} runs, got {len(runs)}")

    missing = []
    for _, row in runs.iterrows():
        run_dir = Path(row["run_dir"])
        checkpoint = Path(row["checkpoint"])
        if not run_dir.is_dir():
            missing.append(f"run_dir: {run_dir}")
        if not checkpoint.is_file():
            missing.append(f"checkpoint: {checkpoint}")
    if missing:
        raise FileNotFoundError(f"{suite_dir.name}: missing paths:\n" + "\n".join(missing[:10]))

    print(f"[ok] {suite_dir.name}: {len(runs)} runs")
    return runs

for protocol in ["clean", "iid", "v4lite"]:
    validate_suite(suite_path("smoke3", protocol), smoke_seeds)

for protocol in ["clean", "iid", "v4lite"]:
    validate_suite(suite_path("decision_extra43-45", protocol), extra_seeds)

for protocol in ["clean", "iid", "v4lite"]:
    smoke_suite = suite_path("smoke3", protocol)
    extra_suite = suite_path("decision_extra43-45", protocol)
    target_suite = ckpt / f"sweep_oc_phase1a_decision_{protocol}_{run_tag}"

    # Safe to remove only this manifest-only/proxy target. Do not remove source suites.
    if target_suite.exists():
        shutil.rmtree(target_suite)

    rows = pd.concat([
        read_runs(smoke_suite).query("seed in @smoke_seeds"),
        read_runs(extra_suite).query("seed in @extra_seeds"),
    ], ignore_index=True)

    if set(rows["seed"]) != decision_seeds:
        raise ValueError(f"{protocol}: decision seed mismatch: {set(rows['seed'])}")
    if len(rows) != len(models) * len(decision_seeds):
        raise ValueError(f"{protocol}: expected 15 rows, got {len(rows)}")

    rows = rows.sort_values(["group", "model_type", "seed"]).copy()

    # Keep absolute pointers to the real run dirs/checkpoints. No symlinks.
    rows["run_dir"] = rows["run_dir"].map(lambda p: str(Path(p).resolve()))
    rows["checkpoint"] = rows["checkpoint"].map(lambda p: str(Path(p).resolve()))

    target_suite.mkdir(parents=True)
    rows.to_csv(target_suite / "runs.tsv", sep="\t", index=False)
    (target_suite / "suite_config.txt").write_text(
        f"Suite: {target_suite.name}\n"
        f"Type: manifest-only decision suite; no symlinks\n"
        f"Sources: smoke3 seeds 42/44/46 + decision_extra43-45 seeds 43/45\n"
        f"Protocol: {protocol}\n",
        encoding="utf-8",
    )

    print(f"[registered] {target_suite.name}: {len(rows)} runs")


[ok] sweep_oc_phase1a_smoke3_clean_phase1a_oc_v4lite_cleanrun_v1: 9 runs
[ok] sweep_oc_phase1a_smoke3_iid_phase1a_oc_v4lite_cleanrun_v1: 9 runs
[ok] sweep_oc_phase1a_smoke3_v4lite_phase1a_oc_v4lite_cleanrun_v1: 9 runs
[ok] sweep_oc_phase1a_decision_extra43-45_clean_phase1a_oc_v4lite_cleanrun_v1: 6 runs
[ok] sweep_oc_phase1a_decision_extra43-45_iid_phase1a_oc_v4lite_cleanrun_v1: 6 runs
[ok] sweep_oc_phase1a_decision_extra43-45_v4lite_phase1a_oc_v4lite_cleanrun_v1: 6 runs
[registered] sweep_oc_phase1a_decision_clean_phase1a_oc_v4lite_cleanrun_v1: 15 runs
[registered] sweep_oc_phase1a_decision_iid_phase1a_oc_v4lite_cleanrun_v1: 15 runs
[registered] sweep_oc_phase1a_decision_v4lite_phase1a_oc_v4lite_cleanrun_v1: 15 runs


In [ ]:
for protocol in ["clean", "iid", "v4lite"]:
    suite = ckpt / f"sweep_oc_phase1a_decision_{protocol}_{run_tag}"
    runs = pd.read_csv(suite / "runs.tsv", sep="\t")
    print(protocol, len(runs), sorted(runs["seed"].unique()))
    print(runs[["model_type", "seed", "run_dir"]].head())

clean 15 [np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46)]
       model_type  seed                                            run_dir
0  ablate_no_lift    42  /content/drive/MyDrive/Colab Notebooks/auvhamn...
1  ablate_no_lift    43  /content/drive/MyDrive/Colab Notebooks/auvhamn...
2  ablate_no_lift    44  /content/drive/MyDrive/Colab Notebooks/auvhamn...
3  ablate_no_lift    45  /content/drive/MyDrive/Colab Notebooks/auvhamn...
4  ablate_no_lift    46  /content/drive/MyDrive/Colab Notebooks/auvhamn...
iid 15 [np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46)]
       model_type  seed                                            run_dir
0  ablate_no_lift    42  /content/drive/MyDrive/Colab Notebooks/auvhamn...
1  ablate_no_lift    43  /content/drive/MyDrive/Colab Notebooks/auvhamn...
2  ablate_no_lift    44  /content/drive/MyDrive/Colab Notebooks/auvhamn...
3  ablate_no_lift    45  /content/drive/MyDrive/Colab Notebooks/auvhamn...
4  ablate_no_lift

In [ ]:
python scripts/phase1a_oc_v4lite_utils.py \
  --checkpoint-root checkpoints \
  audit \
  --suite-name "sweep_oc_phase1a_decision_clean_phase1a_oc_v4lite_cleanrun_v1" \
  --suite-name "sweep_oc_phase1a_decision_iid_phase1a_oc_v4lite_cleanrun_v1" \
  --suite-name "sweep_oc_phase1a_decision_v4lite_phase1a_oc_v4lite_cleanrun_v1" \
  --strict-zero-noise \
  --soft-min-epoch-scale 0.05

In [ ]:
# 审计并验证新增 v4-lite

!python scripts/phase1a_oc_v4lite_utils.py \
  --checkpoint-root checkpoints \
  audit \
  --suite-name "sweep_oc_phase1a_decision_clean_${RUN_TAG}" \
  --suite-name "sweep_oc_phase1a_decision_iid_${RUN_TAG}" \
  --suite-name "sweep_oc_phase1a_decision_v4lite_${RUN_TAG}" \
  --strict-zero-noise \
  --soft-min-epoch-scale 0.05

!python scripts/phase1a_oc_v4lite_utils.py \
  --checkpoint-root checkpoints \
  validate \
  --suite-name "sweep_oc_phase1a_decision_v4lite_${RUN_TAG}"


[audit] /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_clean_phase1a_oc_v4lite_cleanrun_v1
[audit] /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_iid_phase1a_oc_v4lite_cleanrun_v1
[audit] /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_v4lite_phase1a_oc_v4lite_cleanrun_v1
                                                    suite_name    group           model_type  seed train_noise_protocol  best_epoch  best_loss  noise_warmup_epochs  noise_ramp_epochs  noise_mix_ratio  epoch_scale_at_best  is_effectively_clean_at_best
 sweep_oc_phase1a_decision_clean_phase1a_oc_v4lite_cleanrun_v1 ablation       ablate_no_lift    42                clean         249   0.004077                   20                 80              0.5                  0.0                          True
 sweep_oc_phase1a_decision_clean_phase1a_oc_v4lite_cleanrun_v1 ablation       abla

## 7. Decision rollout eval

默认只跑 Phase-1A 必需评估：

- `clean`
- `iid_noisy_ic / nominal_eval`
- `v4_lite / nominal_eval`

`degraded_eval` 和 `heading_biased_eval` 不在默认 gate 内。

In [ ]:
os.environ["MODE"] = "decision_eval"
!bash scripts/run_phase1a_oc_v4lite.sh

Streaming output truncated to the last 5000 lines.
  H=10.0s | pos median=0.0872 m | pos p95=0.2832 m | rot median=0.0056 rad | total vel median=0.0136 m/s | rel vel median=0.0126 m/s | cond_n=30/30 | completion=1.000 | gt_fail_by_h=0.000 | model_fail_by_h=0.000
  H=30.0s | pos median=0.4055 m | pos p95=0.6294 m | rot median=0.0110 rad | total vel median=0.0153 m/s | rel vel median=0.0159 m/s | cond_n=30/30 | completion=1.000 | gt_fail_by_h=0.000 | model_fail_by_h=0.000
  H=60.0s | pos median=1.1320 m | pos p95=2.5938 m | rot median=0.0288 rad | total vel median=0.0164 m/s | rel vel median=0.0148 m/s | cond_n=30/30 | completion=1.000 | gt_fail_by_h=0.000 | model_fail_by_h=0.000

CHIRP
  n=30 | completed=30 (1.000) | gt_divergence=0 (0.000) | pred_divergence=0 (0.000) | solver_failure=0 (0.000) | nan_or_inf=0 (0.000)
  H=10.0s | pos median=0.0944 m | pos p95=0.1610 m | rot median=0.0096 rad | total vel median=0.0071 m/s | rel vel median=0.0059 m/s | cond_n=30/30 | completion=1.000 | gt_

## 8. Decision summarize + export

注册 decision proxy suite，生成并导出 `phase1a_*` 正式产物。

In [ ]:
# unused
"""
os.environ["MODE"] = "decision_summarize"
!bash scripts/run_phase1a_oc_v4lite.sh
"""

In [9]:
from pathlib import Path
import os
import shutil
import pandas as pd

run_tag = os.environ.get("RUN_TAG", "phase1a_oc_v4lite_cleanrun_v1")
ckpt = Path.cwd() / "checkpoints"
target = ckpt / f"sweep_oc_phase1a_decision_proxy_{run_tag}"

if target.exists():
    shutil.rmtree(target)

rows = []
for protocol in ["clean", "iid", "v4lite"]:
    suite = ckpt / f"sweep_oc_phase1a_decision_{protocol}_{run_tag}"
    df = pd.read_csv(suite / "runs.tsv", sep="\t")
    df["seed"] = df["seed"].astype(int)

    if len(df) != 15 or set(df["seed"]) != {42, 43, 44, 45, 46}:
        raise ValueError(f"Invalid decision suite: {suite}")

    for _, row in df.iterrows():
        rows.append({
            "group": row["group"],
            "model_type": row["model_type"],
            "seed": int(row["seed"]),
            "run_name": f"{protocol}__{row['run_name']}",
            "run_dir": str(Path(row["run_dir"]).resolve()),
            "checkpoint": str(Path(row["checkpoint"]).resolve()),
        })

target.mkdir(parents=True)
out = pd.DataFrame(rows).sort_values(["group", "model_type", "seed", "run_name"])
out.to_csv(target / "runs.tsv", sep="\t", index=False)
(target / "suite_config.txt").write_text(
    f"Suite: {target.name}\n"
    "Type: manifest-only combined decision suite; no symlinks\n"
    "Sources: decision_clean + decision_iid + decision_v4lite\n",
    encoding="utf-8",
)

print(target)
print(len(out))


/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_proxy_phase1a_oc_v4lite_cleanrun_v1
45


In [10]:
!python scripts/phase1a_oc_v4lite_utils.py \
  --checkpoint-root checkpoints \
  audit \
  --suite-name "sweep_oc_phase1a_decision_clean_phase1a_oc_v4lite_cleanrun_v1" \
  --suite-name "sweep_oc_phase1a_decision_iid_phase1a_oc_v4lite_cleanrun_v1" \
  --suite-name "sweep_oc_phase1a_decision_v4lite_phase1a_oc_v4lite_cleanrun_v1" \
  --output "checkpoints/sweep_oc_phase1a_decision_proxy_phase1a_oc_v4lite_cleanrun_v1/phase1a_train_audit.csv" \
  --strict-zero-noise \
  --soft-min-epoch-scale 0.05

!python scripts/summarize_sweep.py \
  --suite-dir "checkpoints/sweep_oc_phase1a_decision_proxy_phase1a_oc_v4lite_cleanrun_v1"

!python scripts/build_experiment_report.py \
  --suite-dir "checkpoints/sweep_oc_phase1a_decision_proxy_phase1a_oc_v4lite_cleanrun_v1"


[audit] /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_clean_phase1a_oc_v4lite_cleanrun_v1
[audit] /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_iid_phase1a_oc_v4lite_cleanrun_v1
[audit] /content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_v4lite_phase1a_oc_v4lite_cleanrun_v1
                                                    suite_name    group           model_type  seed train_noise_protocol  best_epoch  best_loss  noise_warmup_epochs  noise_ramp_epochs  noise_mix_ratio  epoch_scale_at_best  is_effectively_clean_at_best
 sweep_oc_phase1a_decision_clean_phase1a_oc_v4lite_cleanrun_v1 ablation       ablate_no_lift    42                clean         249   0.004077                   20                 80              0.5                  0.0                          True
 sweep_oc_phase1a_decision_clean_phase1a_oc_v4lite_cleanrun_v1 ablation       abla

In [ ]:
checkpoints/sweep_oc_phase1a_decision_proxy_phase1a_oc_v4lite_cleanrun_v1/phase1a_summary.csv
checkpoints/sweep_oc_phase1a_decision_proxy_phase1a_oc_v4lite_cleanrun_v1/phase1a_decision_brief.md


## 9. Inspect Phase-1A artifacts

先看 model-level summary，再回到 by-seed / degradation 检查收益是否主要来自单个坏 seed。

In [11]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import Markdown, display

out = Path("checkpoints") / f"sweep_oc_phase1a_decision_proxy_{os.environ['RUN_TAG']}"
print(out)

summary = pd.read_csv(out / "phase1a_summary.csv")
by_seed = pd.read_csv(out / "phase1a_by_seed.csv")
by_horizon = pd.read_csv(out / "phase1a_by_horizon.csv")
degradation = pd.read_csv(out / "phase1a_degradation.csv")
protocol_delta = pd.read_csv(out / "phase1a_protocol_delta.csv")

display(summary.head(20))
display(by_seed.head(20))
display(by_horizon.head(20))
display(degradation.head(20))
display(protocol_delta.head(20))

checkpoints/sweep_oc_phase1a_decision_proxy_phase1a_oc_v4lite_cleanrun_v1


,group,model_type,train_noise_profile,train_noise_protocol,train_protocol_label,source,eval_profile,eval_protocol,eval_protocol_label,horizon_s,...,rollout_final_position_error_p95_min,rollout_final_position_error_p95_max,rollout_final_rotation_geodesic_median_mean,rollout_final_rotation_geodesic_median_std,rollout_final_rotation_geodesic_median_min,rollout_final_rotation_geodesic_median_max,rollout_final_total_linear_velocity_error_median_mean,rollout_final_total_linear_velocity_error_median_std,rollout_final_total_linear_velocity_error_median_min,rollout_final_total_linear_velocity_error_median_max
0,ablation,ablate_no_lift,clean,clean,clean,block,clean,clean,clean,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ablation,ablate_no_lift,clean,clean,clean,block,heading_biased_eval,iid_noisy_ic,iid_noisy_ic,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ablation,ablate_no_lift,clean,clean,clean,block,nominal_eval,iid_noisy_ic,iid_noisy_ic,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ablation,ablate_no_lift,clean,clean,clean,heldout,clean,clean,clean,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ablation,ablate_no_lift,clean,clean,clean,heldout,degraded_eval,iid_noisy_ic,iid_noisy_ic,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ablation,ablate_no_lift,clean,clean,clean,heldout,heading_biased_eval,iid_noisy_ic,iid_noisy_ic,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ablation,ablate_no_lift,clean,clean,clean,heldout,nominal_eval,iid_noisy_ic,iid_noisy_ic,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ablation,ablate_no_lift,clean,clean,clean,rollout,clean,clean,clean,10.0,...,0.123407,7.654650,0.033426,0.054038,0.004809,0.141427,0.090195,0.161193,0.006004,0.412504
8,ablation,ablate_no_lift,clean,clean,clean,rollout,clean,clean,clean,30.0,...,0.527066,29.829862,0.047738,0.072286,0.007539,0.192001,0.173345,0.326877,0.007235,0.827079
9,ablation,ablate_no_lift,clean,clean,clean,rollout,clean,clean,clean,60.0,...,1.278867,74.984107,0.076565,0.114820,0.010432,0.305216,0.248273,0.473799,0.007363,1.195837


,suite_name,group,model_type,seed,run_name,run_dir,dataset_path,dataset_id,noise_reference,best_epoch,...,rollout_final_position_error_median,rollout_final_position_error_p95,rollout_final_rotation_geodesic_median,rollout_final_total_linear_velocity_error_median,source,eval_profile,eval_protocol,eval_protocol_label,horizon_s,summary_path
0,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,d0be9434,remus100_dr,249.0,...,NaN,NaN,NaN,NaN,block,clean,clean,clean,NaN,/content/drive/MyDrive/Colab Notebooks/auvhamn...
1,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,d0be9434,remus100_dr,249.0,...,NaN,NaN,NaN,NaN,block,heading_biased_eval,iid_noisy_ic,iid_noisy_ic,NaN,/content/drive/MyDrive/Colab Notebooks/auvhamn...
2,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,d0be9434,remus100_dr,249.0,...,NaN,NaN,NaN,NaN,block,nominal_eval,iid_noisy_ic,iid_noisy_ic,NaN,/content/drive/MyDrive/Colab Notebooks/auvhamn...
3,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,d0be9434,remus100_dr,249.0,...,NaN,NaN,NaN,NaN,heldout,clean,clean,clean,NaN,/content/drive/MyDrive/Colab Notebooks/auvhamn...
4,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,d0be9434,remus100_dr,249.0,...,NaN,NaN,NaN,NaN,heldout,degraded_eval,iid_noisy_ic,iid_noisy_ic,NaN,/content/drive/MyDrive/Colab Notebooks/auvhamn...
5,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,d0be9434,remus100_dr,249.0,...,NaN,NaN,NaN,NaN,heldout,heading_biased_eval,iid_noisy_ic,iid_noisy_ic,NaN,/content/drive/MyDrive/Colab Notebooks/auvhamn...
6,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,d0be9434,remus100_dr,249.0,...,NaN,NaN,NaN,NaN,heldout,nominal_eval,iid_noisy_ic,iid_noisy_ic,NaN,/content/drive/MyDrive/Colab Notebooks/auvhamn...
7,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,d0be9434,remus100_dr,249.0,...,0.052122,0.123407,0.004883,0.006004,rollout,clean,clean,clean,10.0,/content/drive/MyDrive/Colab Notebooks/auvhamn...
8,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,d0be9434,remus100_dr,249.0,...,0.183027,0.587181,0.007631,0.007574,rollout,clean,clean,clean,30.0,/content/drive/MyDrive/Colab Notebooks/auvhamn...
9,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,d0be9434,remus100_dr,249.0,...,0.526213,1.278867,0.010432,0.007363,rollout,clean,clean,clean,60.0,/content/drive/MyDrive/Colab Notebooks/auvhamn...


,group,model_type,train_noise_profile,train_noise_protocol,train_protocol_label,source,eval_profile,eval_protocol,eval_protocol_label,horizon_s,...,rollout_final_position_error_p95_min,rollout_final_position_error_p95_max,rollout_final_rotation_geodesic_median_mean,rollout_final_rotation_geodesic_median_std,rollout_final_rotation_geodesic_median_min,rollout_final_rotation_geodesic_median_max,rollout_final_total_linear_velocity_error_median_mean,rollout_final_total_linear_velocity_error_median_std,rollout_final_total_linear_velocity_error_median_min,rollout_final_total_linear_velocity_error_median_max
0,ablation,ablate_no_lift,clean,clean,clean,rollout,clean,clean,clean,10.0,...,0.123407,7.654650,0.033426,0.054038,0.004809,0.141427,0.090195,0.161193,0.006004,0.412504
1,ablation,ablate_no_lift,clean,clean,clean,rollout,clean,clean,clean,30.0,...,0.527066,29.829862,0.047738,0.072286,0.007539,0.192001,0.173345,0.326877,0.007235,0.827079
2,ablation,ablate_no_lift,clean,clean,clean,rollout,clean,clean,clean,60.0,...,1.278867,74.984107,0.076565,0.114820,0.010432,0.305216,0.248273,0.473799,0.007363,1.195837
3,ablation,ablate_no_lift,clean,clean,clean,rollout,nominal_eval,iid_noisy_ic,iid_noisy_ic,10.0,...,0.383336,7.609423,0.037581,0.053312,0.010063,0.144193,0.090658,0.161449,0.006797,0.413492
4,ablation,ablate_no_lift,clean,clean,clean,rollout,nominal_eval,iid_noisy_ic,iid_noisy_ic,30.0,...,1.108152,29.952598,0.048844,0.070459,0.010541,0.189577,0.174337,0.327715,0.007871,0.829750
5,ablation,ablate_no_lift,clean,clean,clean,rollout,nominal_eval,iid_noisy_ic,iid_noisy_ic,60.0,...,2.578486,74.656600,0.077385,0.112828,0.014169,0.302244,0.248870,0.474046,0.007893,1.196932
6,ablation,ablate_no_lift,clean,clean,clean,rollout,nominal_eval,v4_lite,v4_lite,10.0,...,0.273539,7.657920,0.037364,0.055756,0.007680,0.148804,0.090199,0.161038,0.006683,0.412219
7,ablation,ablate_no_lift,clean,clean,clean,rollout,nominal_eval,v4_lite,v4_lite,30.0,...,0.932726,29.951731,0.049981,0.072459,0.008976,0.194615,0.173923,0.327585,0.007765,0.829077
8,ablation,ablate_no_lift,clean,clean,clean,rollout,nominal_eval,v4_lite,v4_lite,60.0,...,2.039898,74.782526,0.077244,0.114881,0.012035,0.306215,0.248444,0.473345,0.007252,1.195102
9,ablation,ablate_no_lift,nominal_train,iid_noisy_ic,iid_noisy_ic,rollout,clean,clean,clean,10.0,...,0.215688,0.289163,0.008126,0.002170,0.005610,0.012094,0.012166,0.001644,0.009586,0.013829


,comparison_kind,suite_name,group,model_type,seed,run_name,run_dir,source,metric_name,horizon_s,...,baseline_train_protocol_label,eval_profile,baseline_eval_profile,eval_protocol,baseline_eval_protocol,value,clean_value,absolute_delta,ratio_to_clean,degradation_pct
0,clean_replay_cost,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,iid__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,block,block_angular_rmse_mean,NaN,...,clean,clean,clean,clean,clean,0.001124,0.000602,0.000522,1.866737,86.673683
1,clean_replay_cost,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,iid__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,block,block_position_rmse_mean,NaN,...,clean,clean,clean,clean,clean,0.000155,0.000090,0.000065,1.730366,73.036574
2,clean_replay_cost,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,iid__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,block,block_rotation_geodesic_mean,NaN,...,clean,clean,clean,clean,clean,0.000493,0.000489,0.000004,1.008295,0.829501
3,clean_replay_cost,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,iid__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,block,block_velocity_rmse_mean,NaN,...,clean,clean,clean,clean,clean,0.001803,0.001072,0.000731,1.681408,68.140819
4,clean_replay_cost,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,iid__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,heldout,heldout_angular_rmse_mean,NaN,...,clean,clean,clean,clean,clean,0.000951,0.000512,0.000439,1.856780,85.677999
5,clean_replay_cost,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,iid__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,heldout,heldout_position_rmse_mean,NaN,...,clean,clean,clean,clean,clean,0.000126,0.000071,0.000055,1.779302,77.930172
6,clean_replay_cost,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,iid__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,heldout,heldout_rotation_geodesic_mean,NaN,...,clean,clean,clean,clean,clean,0.000491,0.000489,0.000002,1.005086,0.508550
7,clean_replay_cost,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,iid__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,heldout,heldout_success_rate,NaN,...,clean,clean,clean,clean,clean,1.000000,1.000000,0.000000,1.000000,0.000000
8,clean_replay_cost,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,iid__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,heldout,heldout_velocity_rmse_mean,NaN,...,clean,clean,clean,clean,clean,0.001457,0.000839,0.000617,1.735873,73.587295
9,clean_replay_cost,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,iid__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,rollout,rollout_completion_rate,10.0,...,clean,clean,clean,clean,clean,1.000000,1.000000,0.000000,1.000000,0.000000


,comparison_kind,table_scope,suite_name,group,model_type,seed,run_name,baseline_run_name,run_dir,baseline_run_dir,...,baseline_eval_protocol_label,summary_path,baseline_summary_path,metric_name,value,baseline_value,absolute_delta,ratio_to_baseline,degradation_pct,improvement_pct
0,eval_protocol_v4_lite_vs_iid_noisy_ic,by_scenario,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,...,iid_noisy_ic,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,rollout_completion_rate,1.000000,1.000000,0.000000,1.000000,0.000000,-0.000000
1,eval_protocol_v4_lite_vs_iid_noisy_ic,by_scenario,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,...,iid_noisy_ic,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,rollout_completion_rate,1.000000,1.000000,0.000000,1.000000,0.000000,-0.000000
2,eval_protocol_v4_lite_vs_iid_noisy_ic,by_scenario,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,...,iid_noisy_ic,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,rollout_completion_rate,1.000000,1.000000,0.000000,1.000000,0.000000,-0.000000
3,eval_protocol_v4_lite_vs_iid_noisy_ic,by_scenario,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,...,iid_noisy_ic,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,rollout_final_position_error_median,0.103742,0.125072,-0.021330,0.829460,-17.053991,17.053991
4,eval_protocol_v4_lite_vs_iid_noisy_ic,by_scenario,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,...,iid_noisy_ic,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,rollout_final_position_error_median,0.353640,0.383184,-0.029544,0.922899,-7.710114,7.710114
5,eval_protocol_v4_lite_vs_iid_noisy_ic,by_scenario,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,...,iid_noisy_ic,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,rollout_final_position_error_median,0.697540,0.777395,-0.079855,0.897278,-10.272184,10.272184
6,eval_protocol_v4_lite_vs_iid_noisy_ic,by_scenario,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lift_seed42,clean__ablation_ablate_no_lift_seed42,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,...,iid_noisy_ic,/content/drive/MyDrive/Colab Notebooks/auvhamn...,/content/drive/MyDrive/Colab Notebooks/auvhamn...,rollout_final_position_error_p95,0.278621,0.326892,-0.048272,0.852331,-14.766881,14.766881
7,eval_protocol_v4_lite_vs_iid_noisy_ic,by_scenario,sweep_oc_phase1a_decision_proxy_phase1a_oc_v4l...,ablation,ablate_no_lift,42,clean__ablation_ablate_no_lif

In [12]:
brief = Path("checkpoints") / f"sweep_oc_phase1a_decision_proxy_{os.environ['RUN_TAG']}" / "phase1a_decision_brief.md"
display(Markdown(brief.read_text()))

# Phase-1A Decision Brief

- Suite: `/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7/checkpoints/sweep_oc_phase1a_decision_proxy_phase1a_oc_v4lite_cleanrun_v1`
- Generated: `2026-04-26 15:54:03`
- Runs: `45`
- Horizons: `10s, 30s, 60s`
- Primary rollout profile: `nominal_eval`
- Primary heldout profile: `nominal_eval`
- Dataset: `/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_5/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl`

## Key Findings

- Best rollout aggregate at `60s` / `nominal_eval` is `main/phnode_full` under train protocol `clean` and eval protocol `v4_lite`, with `pos median=0.8439 m` and `completion=99.1%`.
- `phnode_full` still shows seed sensitivity at `60s`: position-median range across matching rows is `0.7852 m`.
- Clean replay cost rows are available for `6` non-clean training groups, so the report can distinguish robustness gains from clean-performance regressions.

## Scope

- Scope: this Phase-1A decision brief covers the OC known-current surrogate tier only, which is the intended Phase-1A data layer.

## Rollout Summary @60s / nominal_eval

| Model | Train | Eval | Seeds | Pos Median | Pos P95 | Completion | Model Fail |
| --- | --- | --- | --- | --- | --- | --- | --- |
| main/phnode_full | clean | v4_lite | 42,43,44,45,46 | 0.8439 | 2.1544 | 99.1% | 0.9% |
| main/phnode_full | clean | iid_noisy_ic | 42,43,44,45,46 | 0.9604 | 2.8640 | 99.1% | 0.9% |
| ablation/ablate_no_lift | iid_noisy_ic/nominal_train | iid_noisy_ic | 42,43,44,45,46 | 1.0043 | 3.4515 | 98.2% | 1.8% |
| main/phnode_full | iid_noisy_ic/nominal_train | v4_lite | 42,43,44,45,46 | 1.0317 | 2.9404 | 98.4% | 1.6% |
| ablation/ablate_no_lift | iid_noisy_ic/nominal_train | v4_lite | 42,43,44,45,46 | 1.0551 | 3.0847 | 98.2% | 1.8% |
| main/phnode_full | iid_noisy_ic/nominal_train | iid_noisy_ic | 42,43,44,45,46 | 1.0934 | 3.2639 | 98.2% | 1.8% |
| main/phnode_full | v4_lite/nominal_train | iid_noisy_ic | 42,43,44,45,46 | 1.1116 | 3.3398 | 98.2% | 1.8% |
| main/phnode_full | v4_lite/nominal_train | v4_lite | 42,43,44,45,46 | 1.1148 | 2.9198 | 98.2% | 1.8% |
| ablation/ablate_no_lift | v4_lite/nominal_train | iid_noisy_ic | 42,43,44,45,46 | 1.2696 | 3.6817 | 98.4% | 1.6% |
| ablation/ablate_no_lift | v4_lite/nominal_train | v4_lite | 42,43,44,45,46 | 1.3078 | 3.5389 | 98.4% | 1.6% |
| ablation/ablate_no_mass_prior | v4_lite/nominal_train | iid_noisy_ic | 42,43,44,45,46 | 1.4021 | 4.5957 | 98.2% | 1.8% |
| ablation/ablate_no_mass_prior | v4_lite/nominal_train | v4_lite | 42,43,44,45,46 | 1.4075 | 4.0668 | 98.4% | 1.6% |
| ablation/ablate_no_mass_prior | clean | iid_noisy_ic | 42,43,44,45,46 | 1.4163 | 4.6889 | 98.7% | 1.3% |
| ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | iid_noisy_ic | 42,43,44,45,46 | 1.4613 | 4.3018 | 98.2% | 1.8% |
| ablation/ablate_no_mass_prior | clean | v4_lite | 42,43,44,45,46 | 1.4622 | 4.2905 | 98.7% | 1.3% |
| ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | v4_lite | 42,43,44,45,46 | 1.5156 | 3.8697 | 98.2% | 1.8% |
| ablation/ablate_no_lift | clean | v4_lite | 42,43,44,45,46 | 9.6808 | 17.1002 | 98.9% | 1.1% |
| ablation/ablate_no_lift | clean | iid_noisy_ic | 42,43,44,45,46 | 9.7397 | 17.4667 | 98.9% | 1.1% |

## Heldout Summary / nominal_eval

| Model | Train | Seeds | Pos RMSE Mean | Rot Geo Mean | Success |
| --- | --- | --- | --- | --- | --- |
| main/phnode_full | clean | 42,44,46 | 0.00111 | 0.00860 | 100.0% |
| ablation/ablate_no_lift | clean | 42,44,46 | 0.00111 | 0.00861 | 100.0% |
| ablation/ablate_no_lift | iid_noisy_ic/nominal_train | 42,44,46 | 0.00111 | 0.00862 | 100.0% |
| main/phnode_full | iid_noisy_ic/nominal_train | 42,44,46 | 0.00111 | 0.00862 | 100.0% |
| ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | 42,44,46 | 0.00111 | 0.00862 | 100.0% |
| ablation/ablate_no_mass_prior | clean | 42,44,46 | 0.00111 | 0.00861 | 100.0% |
| ablation/ablate_no_mass_prior | v4_lite/nominal_train | 42,44,46 | 0.00111 | 0.00869 | 100.0% |
| main/phnode_full | v4_lite/nominal_train | 42,44,46 | 0.00111 | 0.00869 | 100.0% |
| ablation/ablate_no_lift | v4_lite/nominal_train | 42,44,46 | 0.00112 | 0.00869 | 100.0% |

## Rollout By Horizon / nominal_eval

| Model | Train | Eval | Pos @10s | Pos @30s | Pos @60s | Completion @60s |
| --- | --- | --- | --- | --- | --- | --- |
| main/phnode_full | clean | v4_lite | 0.1348 | 0.3824 | 0.8439 | 99.1% |
| main/phnode_full | clean | iid_noisy_ic | 0.1568 | 0.4188 | 0.9604 | 99.1% |
| ablation/ablate_no_lift | iid_noisy_ic/nominal_train | iid_noisy_ic | 0.1669 | 0.4582 | 1.0043 | 98.2% |
| main/phnode_full | iid_noisy_ic/nominal_train | v4_lite | 0.1474 | 0.4492 | 1.0317 | 98.4% |
| ablation/ablate_no_lift | iid_noisy_ic/nominal_train | v4_lite | 0.1620 | 0.4538 | 1.0551 | 98.2% |
| main/phnode_full | iid_noisy_ic/nominal_train | iid_noisy_ic | 0.1697 | 0.4716 | 1.0934 | 98.2% |
| main/phnode_full | v4_lite/nominal_train | iid_noisy_ic | 0.1620 | 0.4521 | 1.1116 | 98.2% |
| main/phnode_full | v4_lite/nominal_train | v4_lite | 0.1520 | 0.4604 | 1.1148 | 98.2% |
| ablation/ablate_no_lift | v4_lite/nominal_train | iid_noisy_ic | 0.1703 | 0.4929 | 1.2696 | 98.4% |
| ablation/ablate_no_lift | v4_lite/nominal_train | v4_lite | 0.1632 | 0.4875 | 1.3078 | 98.4% |
| ablation/ablate_no_mass_prior | v4_lite/nominal_train | iid_noisy_ic | 0.1900 | 0.6120 | 1.4021 | 98.2% |
| ablation/ablate_no_mass_prior | v4_lite/nominal_train | v4_lite | 0.1779 | 0.6009 | 1.4075 | 98.4% |
| ablation/ablate_no_mass_prior | clean | iid_noisy_ic | 0.1954 | 0.6255 | 1.4163 | 98.7% |
| ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | iid_noisy_ic | 0.1872 | 0.5760 | 1.4613 | 98.2% |
| ablation/ablate_no_mass_prior | clean | v4_lite | 0.1874 | 0.6525 | 1.4622 | 98.7% |
| ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | v4_lite | 0.1781 | 0.5886 | 1.5156 | 98.2% |
| ablation/ablate_no_lift | clean | v4_lite | 0.7506 | 3.2548 | 9.6808 | 98.9% |
| ablation/ablate_no_lift | clean | iid_noisy_ic | 0.7881 | 3.3884 | 9.7397 | 98.9% |

## Rollout By Scenario @60s / nominal_eval

| Model | Train | Eval | Scenario | Completion | Model Fail | Pos Median | Pos P95 | Rot Median |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| ablation/ablate_no_lift | clean | iid_noisy_ic | CHIRP | 100.0% | 0.0% | 9.7736 | 12.1483 | 0.0590 |
| ablation/ablate_no_lift | clean | iid_noisy_ic | OU | 96.7% | 3.3% | 10.3707 | 17.3041 | 0.1854 |
| ablation/ablate_no_lift | clean | iid_noisy_ic | PRBS | 100.0% | 0.0% | 8.8025 | 18.9399 | 0.0594 |
| ablation/ablate_no_lift | clean | v4_lite | CHIRP | 100.0% | 0.0% | 9.8137 | 12.4841 | 0.0583 |
| ablation/ablate_no_lift | clean | v4_lite | OU | 96.7% | 3.3% | 10.1888 | 16.5584 | 0.1820 |
| ablation/ablate_no_lift | clean | v4_lite | PRBS | 100.0% | 0.0% | 8.6724 | 18.2717 | 0.0587 |
| ablation/ablate_no_lift | iid_noisy_ic/nominal_train | iid_noisy_ic | CHIRP | 100.0% | 0.0% | 0.8016 | 2.3218 | 0.0128 |
| ablation/ablate_no_lift | iid_noisy_ic/nominal_train | iid_noisy_ic | OU | 94.7% | 5.3% | 1.4331 | 4.2132 | 0.0330 |
| ablation/ablate_no_lift | iid_noisy_ic/nominal_train | iid_noisy_ic | PRBS | 100.0% | 0.0% | 0.8955 | 2.9182 | 0.0154 |
| ablation/ablate_no_lift | iid_noisy_ic/nominal_train | v4_lite | CHIRP | 100.0% | 0.0% | 0.9594 | 2.3745 | 0.0125 |
| ablation/ablate_no_lift | iid_noisy_ic/nominal_train | v4_lite | OU | 94.7% | 5.3% | 1.2316 | 3.8106 | 0.0308 |
| ablation/ablate_no_lift | iid_noisy_ic/nominal_train | v4_lite | PRBS | 100.0% | 0.0% | 0.8889 | 2.7336 | 0.0180 |
| ablation/ablate_no_lift | v4_lite/nominal_train | iid_noisy_ic | CHIRP | 100.0% | 0.0% | 1.4289 | 2.8832 | 0.0326 |
| ablation/ablate_no_lift | v4_lite/nominal_train | iid_noisy_ic | OU | 95.3% | 4.7% | 1.4524 | 3.8966 | 0.0343 |
| ablation/ablate_no_lift | v4_lite/nominal_train | iid_noisy_ic | PRBS | 100.0% | 0.0% | 1.0962 | 3.2329 | 0.0235 |
| ablation/ablate_no_lift | v4_lite/nominal_train | v4_lite | CHIRP | 100.0% | 0.0% | 1.4273 | 3.0853 | 0.0335 |
| ablation/ablate_no_lift | v4_lite/nominal_train | v4_lite | OU | 95.3% | 4.7% | 1.2267 | 3.7117 | 0.0299 |
| ablation/ablate_no_lift | v4_lite/nominal_train | v4_lite | PRBS | 100.0% | 0.0% | 1.2443 | 3.0464 | 0.0257 |
| ablation/ablate_no_mass_prior | clean | iid_noisy_ic | CHIRP | 100.0% | 0.0% | 1.2412 | 3.0954 | 0.0214 |
| ablation/ablate_no_mass_prior | clean | iid_noisy_ic | OU | 96.0% | 4.0% | 2.0360 | 5.4938 | 0.0502 |
| ablation/ablate_no_mass_prior | clean | iid_noisy_ic | PRBS | 100.0% | 0.0% | 1.3624 | 4.4595 | 0.0280 |
| ablation/ablate_no_mass_prior | clean | v4_lite | CHIRP | 100.0% | 0.0% | 1.1182 | 3.2445 | 0.0205 |
| ablation/ablate_no_mass_prior | clean | v4_lite | OU | 96.0% | 4.0% | 2.0406 | 4.7608 | 0.0514 |
| ablation/ablate_no_mass_prior | clean | v4_lite | PRBS | 100.0% | 0.0% | 1.5260 | 4.0805 | 0.0309 |
| ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | iid_noisy_ic | CHIRP | 100.0% | 0.0% | 1.2244 | 3.0779 | 0.0263 |
| ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | iid_noisy_ic | OU | 94.7% | 5.3% | 1.6851 | 4.3770 | 0.0359 |
| ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | iid_noisy_ic | PRBS | 100.0% | 0.0% | 1.6672 | 4.4856 | 0.0315 |
| ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | v4_lite | CHIRP | 100.0% | 0.0% | 1.3528 | 3.0622 | 0.0271 |
| ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | v4_lite | OU | 94.7% | 5.3% | 1.5726 | 3.8334 | 0.0356 |
| ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | v4_lite | PRBS | 100.0% | 0.0% | 1.5924 | 3.8341 | 0.0336 |
| ablation/ablate_no_mass_prior | v4_lite/nominal_train | iid_noisy_ic | CHIRP | 100.0% | 0.0% | 1.0343 | 2.7278 | 0.0229 |
| ablation/ablate_no_mass_prior | v4_lite/nominal_train | iid_noisy_ic | OU | 94.7% | 5.3% | 1.9713 | 5.9156 | 0.0498 |
| ablation/ablate_no_mass_prior | v4_lite/nominal_train | iid_noisy_ic | PRBS | 100.0% | 0.0% | 1.3482 | 3.5786 | 0.0267 |
| ablation/ablate_no_mass_prior | v4_lite/nominal_train | v4_lite | CHIRP | 100.0% | 0.0% | 1.0807 | 2.8002 | 0.0185 |
| ablation/ablate_no_mass_prior | v4_lite/nominal_train | v4_lite | OU | 95.3% | 4.7% | 2.0894 | 5.0300 | 0.0497 |
| ablation/ablate_no_mass_prior | v4_lite/nominal_train | v4_lite | PRBS | 100.0% | 0.0% | 1.3557 | 3.0738 | 0.0281 |
| main/phnode_full | clean | iid_noisy_ic | CHIRP | 100.0% | 0.0% | 0.8862 | 2.0746 | 0.0139 |
| main/phnode_full | clean | iid_noisy_ic | OU | 97.3% | 2.7% | 1.0076 | 3.2972 | 0.0269 |
| main/phnode_full | clean | iid_noisy_ic | PRBS | 100.0% | 0.0% | 0.9578 | 2.8805 | 0.0157 |
| main/phnode_full | clean | v4_lite | CHIRP | 100.0% | 0.0% | 0.8158 | 2.1192 | 0.0148 |
| main/phnode_full | clean | v4_lite | OU | 97.3% | 2.7% | 0.9017 | 2.5277 | 0.0186 |
| main/phnode_full | clean | v4_lite | PRBS | 100.0% | 0.0% | 0.7771 | 1.8566 | 0.0135 |
| main/phnode_full | iid_noisy_ic/nominal_train | iid_noisy_ic | CHIRP | 100.0% | 0.0% | 0.9667 | 2.3579 | 0.0172 |
| main/phnode_full | iid_noisy_ic/nominal_train | iid_noisy_ic | OU | 94.7% | 5.3% | 1.2296 | 3.5567 | 0.0300 |
| main/phnode_full | iid_noisy_ic/nominal_train | iid_noisy_ic | PRBS | 100.0% | 0.0% | 1.1550 | 3.3646 | 0.0227 |
| main/phnode_full | iid_noisy_ic/nominal_train | v4_lite | CHIRP | 100.0% | 0.0% | 0.9606 | 2.4210 | 0.0163 |
| main/phnode_full | iid_noisy_ic/nominal_train | v4_lite | OU | 95.3% | 4.7% | 1.1224 | 3.3352 | 0.0251 |
| main/phnode_full | iid_noisy_ic/nominal_train | v4_lite | PRBS | 100.0% | 0.0% | 1.0682 | 2.5191 | 0.0220 |
| main/phnode_full | v4_lite/nominal_train | iid_noisy_ic | CHIRP | 100.0% | 0.0% | 1.0175 | 2.7701 | 0.0222 |
| main/phnode_full | v4_lite/nominal_train | iid_noisy_ic | OU | 94.7% | 5.3% | 1.2898 | 3.8120 | 0.0316 |
| main/phnode_full | v4_lite/nominal_train | iid_noisy_ic | PRBS | 100.0% | 0.0% | 1.0187 | 2.8829 | 0.0201 |
| main/phnode_full | v4_lite/nominal_train | v4_lite | CHIRP | 100.0% | 0.0% | 1.1222 | 2.5319 | 0.0225 |
| main/phnode_full | v4_lite/nominal_train | v4_lite | OU | 94.7% | 5.3% | 1.1165 | 3.0737 | 0.0283 |
| main/phnode_full | v4_lite/nominal_train | v4_lite | PRBS | 100.0% | 0.0% | 1.0535 | 2.4751 | 0.0188 |

## Seed-Level Rollout @60s / nominal_eval

| Run | Model | Train | Eval | Seed | Pos Median | Pos P95 | Completion | Model Fail |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| clean__main_phnode_full_seed46 | main/phnode_full | clean | v4_lite | 46 | 0.6618 | 1.8690 | 100.0% | 0.0% |
| clean__main_phnode_full_seed45 | main/phnode_full | clean | v4_lite | 45 | 0.6721 | 1.8533 | 100.0% | 0.0% |
| clean__main_phnode_full_seed43 | main/phnode_full | clean | v4_lite | 43 | 0.6888 | 1.9741 | 100.0% | 0.0% |
| clean__ablation_ablate_no_lift_seed42 | ablation/ablate_no_lift | clean | v4_lite | 42 | 0.7324 | 2.0399 | 97.8% | 2.2% |
| clean__ablation_ablate_no_lift_seed45 | ablation/ablate_no_lift | clean | v4_lite | 45 | 0.7747 | 2.2748 | 98.9% | 1.1% |
| clean__main_phnode_full_seed42 | main/phnode_full | clean | iid_noisy_ic | 42 | 0.8095 | 2.6315 | 97.8% | 2.2% |
| clean__main_phnode_full_seed45 | main/phnode_full | clean | iid_noisy_ic | 45 | 0.8109 | 2.6622 | 100.0% | 0.0% |
| clean__ablation_ablate_no_lift_seed42 | ablation/ablate_no_lift | clean | iid_noisy_ic | 42 | 0.8114 | 2.5833 | 97.8% | 2.2% |
| iid__main_phnode_full_seed45 | main/phnode_full | iid_noisy_ic/nominal_train | v4_lite | 45 | 0.8194 | 2.2673 | 98.9% | 1.1% |
| clean__main_phnode_full_seed46 | main/phnode_full | clean | iid_noisy_ic | 46 | 0.8203 | 2.5168 | 100.0% | 0.0% |
| iid__ablation_ablate_no_lift_seed44 | ablation/ablate_no_lift | iid_noisy_ic/nominal_train | iid_noisy_ic | 44 | 0.8266 | 3.0526 | 97.8% | 2.2% |
| clean__ablation_ablate_no_lift_seed45 | ablation/ablate_no_lift | clean | iid_noisy_ic | 45 | 0.8609 | 2.5785 | 98.9% | 1.1% |
| v4lite__ablation_ablate_no_lift_seed45 | ablation/ablate_no_lift | v4_lite/nominal_train | v4_lite | 45 | 0.8664 | 2.8093 | 97.8% | 2.2% |
| clean__main_phnode_full_seed42 | main/phnode_full | clean | v4_lite | 42 | 0.8684 | 1.9885 | 97.8% | 2.2% |
| v4lite__ablation_ablate_no_mass_prior_seed42 | ablation/ablate_no_mass_prior | v4_lite/nominal_train | iid_noisy_ic | 42 | 0.8923 | 3.2552 | 98.9% | 1.1% |
| v4lite__ablation_ablate_no_lift_seed45 | ablation/ablate_no_lift | v4_lite/nominal_train | iid_noisy_ic | 45 | 0.8990 | 3.2726 | 97.8% | 2.2% |
| clean__main_phnode_full_seed43 | main/phnode_full | clean | iid_noisy_ic | 43 | 0.9145 | 2.7506 | 100.0% | 0.0% |
| v4lite__ablation_ablate_no_lift_seed44 | ablation/ablate_no_lift | v4_lite/nominal_train | v4_lite | 44 | 0.9273 | 3.0737 | 98.9% | 1.1% |
| v4lite__main_phnode_full_seed43 | main/phnode_full | v4_lite/nominal_train | v4_lite | 43 | 0.9436 | 2.3235 | 97.8% | 2.2% |
| iid__main_phnode_full_seed46 | main/phnode_full | iid_noisy_ic/nominal_train | v4_lite | 46 | 0.9469 | 2.9624 | 97.8% | 2.2% |
| v4lite__ablation_ablate_no_mass_prior_seed42 | ablation/ablate_no_mass_prior | v4_lite/nominal_train | v4_lite | 42 | 0.9540 | 2.7648 | 98.9% | 1.1% |
| iid__main_phnode_full_seed43 | main/phnode_full | iid_noisy_ic/nominal_train | iid_noisy_ic | 43 | 0.9545 | 3.2740 | 97.8% | 2.2% |
| v4lite__ablation_ablate_no_lift_seed42 | ablation/ablate_no_lift | v4_lite/nominal_train | iid_noisy_ic | 42 | 0.9641 | 3.4543 | 97.8% | 2.2% |
| iid__main_phnode_full_seed43 | main/phnode_full | iid_noisy_ic/nominal_train | v4_lite | 43 | 0.9797 | 2.7632 | 97.8% | 2.2% |
| v4lite__main_phnode_full_seed43 | main/phnode_full | v4_lite/nominal_train | iid_noisy_ic | 43 | 0.9868 | 2.6973 | 97.8% | 2.2% |
| iid__ablation_ablate_no_lift_seed43 | ablation/ablate_no_lift | iid_noisy_ic/nominal_train | iid_noisy_ic | 43 | 0.9870 | 3.0326 | 97.8% | 2.2% |
| iid__ablation_ablate_no_lift_seed44 | ablation/ablate_no_lift | iid_noisy_ic/nominal_train | v4_lite | 44 | 0.9885 | 2.5788 | 97.8% | 2.2% |
| iid__ablation_ablate_no_mass_prior_seed42 | ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | iid_noisy_ic | 42 | 0.9981 | 3.2458 | 98.9% | 1.1% |
| v4lite__main_phnode_full_seed42 | main/phnode_full | v4_lite/nominal_train | iid_noisy_ic | 42 | 1.0064 | 3.3526 | 98.9% | 1.1% |
| iid__main_phnode_full_seed42 | main/phnode_full | iid_noisy_ic/nominal_train | iid_noisy_ic | 42 | 1.0081 | 4.2947 | 98.9% | 1.1% |
| iid__ablation_ablate_no_lift_seed45 | ablation/ablate_no_lift | iid_noisy_ic/nominal_train | iid_noisy_ic | 45 | 1.0093 | 3.1251 | 97.8% | 2.2% |
| clean__ablation_ablate_no_lift_seed44 | ablation/ablate_no_lift | clean | v4_lite | 44 | 1.0118 | 2.6263 | 98.9% | 1.1% |
| v4lite__main_phnode_full_seed42 | main/phnode_full | v4_lite/nominal_train | v4_lite | 42 | 1.0205 | 2.9773 | 98.9% | 1.1% |
| clean__ablation_ablate_no_lift_seed44 | ablation/ablate_no_lift | clean | iid_noisy_ic | 44 | 1.0208 | 2.8858 | 98.9% | 1.1% |
| iid__ablation_ablate_no_mass_prior_seed42 | ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | v4_lite | 42 | 1.0270 | 3.0243 | 98.9% | 1.1% |
| v4lite__ablation_ablate_no_lift_seed43 | ablation/ablate_no_lift | v4_lite/nominal_train | iid_noisy_ic | 43 | 1.0314 | 3.3823 | 98.9% | 1.1% |
| iid__ablation_ablate_no_lift_seed42 | ablation/ablate_no_lift | iid_noisy_ic/nominal_train | v4_lite | 42 | 1.0353 | 3.4581 | 97.8% | 2.2% |
| iid__ablation_ablate_no_lift_seed43 | ablation/ablate_no_lift | iid_noisy_ic/nominal_train | v4_lite | 43 | 1.0417 | 2.6931 | 97.8% | 2.2% |
| iid__main_phnode_full_seed45 | main/phnode_full | iid_noisy_ic/nominal_train | iid_noisy_ic | 45 | 1.0639 | 2.5984 | 97.8% | 2.2% |
| iid__main_phnode_full_seed44 | main/phnode_full | iid_noisy_ic/nominal_train | v4_lite | 44 | 1.0656 | 3.2083 | 98.9% | 1.1% |
| iid__ablation_ablate_no_lift_seed45 | ablation/ablate_no_lift | iid_noisy_ic/nominal_train | v4_lite | 45 | 1.0756 | 3.2832 | 97.8% | 2.2% |
| clean__ablation_ablate_no_mass_prior_seed42 | ablation/ablate_no_mass_prior | clean | v4_lite | 42 | 1.0782 | 3.5954 | 100.0% | 0.0% |
| v4lite__main_phnode_full_seed44 | main/phnode_full | v4_lite/nominal_train | iid_noisy_ic | 44 | 1.0800 | 3.7178 | 98.9% | 1.1% |
| clean__ablation_ablate_no_mass_prior_seed46 | ablation/ablate_no_mass_prior | clean | iid_noisy_ic | 46 | 1.0838 | 3.3977 | 97.8% | 2.2% |
| v4lite__main_phnode_full_seed44 | main/phnode_full | v4_lite/nominal_train | v4_lite | 44 | 1.0878 | 3.4630 | 98.9% | 1.1% |
| v4lite__main_phnode_full_seed46 | main/phnode_full | v4_lite/nominal_train | iid_noisy_ic | 46 | 1.0899 | 3.5875 | 97.8% | 2.2% |
| iid__ablation_ablate_no_lift_seed42 | ablation/ablate_no_lift | iid_noisy_ic/nominal_train | iid_noisy_ic | 42 | 1.0938 | 3.7961 | 97.8% | 2.2% |
| iid__ablation_ablate_no_lift_seed46 | ablation/ablate_no_lift | iid_noisy_ic/nominal_train | iid_noisy_ic | 46 | 1.1047 | 4.2513 | 100.0% | 0.0% |
| clean__ablation_ablate_no_mass_prior_seed42 | ablation/ablate_no_mass_prior | clean | iid_noisy_ic | 42 | 1.1083 | 3.4585 | 100.0% | 0.0% |
| v4lite__ablation_ablate_no_lift_seed44 | ablation/ablate_no_lift | v4_lite/nominal_train | iid_noisy_ic | 44 | 1.1309 | 2.8564 | 98.9% | 1.1% |
| clean__ablation_ablate_no_mass_prior_seed46 | ablation/ablate_no_mass_prior | clean | v4_lite | 46 | 1.1333 | 2.9964 | 97.8% | 2.2% |
| iid__ablation_ablate_no_lift_seed46 | ablation/ablate_no_lift | iid_noisy_ic/nominal_train | v4_lite | 46 | 1.1342 | 3.4102 | 100.0% | 0.0% |
| v4lite__ablation_ablate_no_lift_seed43 | ablation/ablate_no_lift | v4_lite/nominal_train | v4_lite | 43 | 1.1581 | 2.7323 | 98.9% | 1.1% |
| iid__main_phnode_full_seed44 | main/phnode_full | iid_noisy_ic/nominal_train | iid_noisy_ic | 44 | 1.1830 | 3.3477 | 98.9% | 1.1% |
| iid__ablation_ablate_no_mass_prior_seed46 | ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | v4_lite | 46 | 1.1831 | 2.9045 | 97.8% | 2.2% |
| v4lite__ablation_ablate_no_lift_seed42 | ablation/ablate_no_lift | v4_lite/nominal_train | v4_lite | 42 | 1.1887 | 3.3797 | 97.8% | 2.2% |
| iid__ablation_ablate_no_mass_prior_seed46 | ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | iid_noisy_ic | 46 | 1.1901 | 3.9359 | 97.8% | 2.2% |
| v4lite__main_phnode_full_seed45 | main/phnode_full | v4_lite/nominal_train | v4_lite | 45 | 1.2333 | 2.8957 | 97.8% | 2.2% |
| v4lite__ablation_ablate_no_mass_prior_seed44 | ablation/ablate_no_mass_prior | v4_lite/nominal_train | v4_lite | 44 | 1.2335 | 5.1703 | 97.8% | 2.2% |
| iid__main_phnode_full_seed46 | main/phnode_full | iid_noisy_ic/nominal_train | iid_noisy_ic | 46 | 1.2572 | 2.8045 | 97.8% | 2.2% |
| v4lite__main_phnode_full_seed46 | main/phnode_full | v4_lite/nominal_train | v4_lite | 46 | 1.2890 | 2.9396 | 97.8% | 2.2% |
| clean__main_phnode_full_seed44 | main/phnode_full | clean | v4_lite | 44 | 1.3284 | 3.0870 | 97.8% | 2.2% |
| iid__main_phnode_full_seed42 | main/phnode_full | iid_noisy_ic/nominal_train | v4_lite | 42 | 1.3471 | 3.5008 | 98.9% | 1.1% |
| iid__ablation_ablate_no_mass_prior_seed44 | ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | v4_lite | 44 | 1.3482 | 3.6683 | 98.9% | 1.1% |
| v4lite__main_phnode_full_seed45 | main/phnode_full | v4_lite/nominal_train | iid_noisy_ic | 45 | 1.3948 | 3.3436 | 97.8% | 2.2% |
| v4lite__ablation_ablate_no_mass_prior_seed45 | ablation/ablate_no_mass_prior | v4_lite/nominal_train | v4_lite | 45 | 1.4082 | 4.5822 | 98.9% | 1.1% |
| v4lite__ablation_ablate_no_mass_prior_seed44 | ablation/ablate_no_mass_prior | v4_lite/nominal_train | iid_noisy_ic | 44 | 1.4175 | 5.8993 | 97.8% | 2.2% |
| clean__main_phnode_full_seed44 | main/phnode_full | clean | iid_noisy_ic | 44 | 1.4469 | 3.7588 | 97.8% | 2.2% |
| v4lite__ablation_ablate_no_mass_prior_seed45 | ablation/ablate_no_mass_prior | v4_lite/nominal_train | iid_noisy_ic | 45 | 1.4910 | 5.0465 | 97.8% | 2.2% |
| clean__ablation_ablate_no_lift_seed46 | ablation/ablate_no_lift | clean | v4_lite | 46 | 1.4920 | 3.7774 | 98.9% | 1.1% |
| clean__ablation_ablate_no_mass_prior_seed44 | ablation/ablate_no_mass_prior | clean | v4_lite | 44 | 1.5017 | 4.6016 | 98.9% | 1.1% |
| clean__ablation_ablate_no_mass_prior_seed45 | ablation/ablate_no_mass_prior | clean | iid_noisy_ic | 45 | 1.5033 | 4.6417 | 97.8% | 2.2% |
| clean__ablation_ablate_no_mass_prior_seed44 | ablation/ablate_no_mass_prior | clean | iid_noisy_ic | 44 | 1.5083 | 5.2205 | 98.9% | 1.1% |
| clean__ablation_ablate_no_lift_seed46 | ablation/ablate_no_lift | clean | iid_noisy_ic | 46 | 1.5102 | 4.6293 | 98.9% | 1.1% |
| clean__ablation_ablate_no_mass_prior_seed45 | ablation/ablate_no_mass_prior | clean | v4_lite | 45 | 1.5323 | 3.8778 | 97.8% | 2.2% |
| v4lite__ablation_ablate_no_mass_prior_seed46 | ablation/ablate_no_mass_prior | v4_lite/nominal_train | iid_noisy_ic | 46 | 1.5712 | 4.6298 | 97.8% | 2.2% |
| iid__ablation_ablate_no_mass_prior_seed44 | ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | iid_noisy_ic | 44 | 1.6001 | 4.1894 | 98.9% | 1.1% |
| v4lite__ablation_ablate_no_mass_prior_seed46 | ablation/ablate_no_mass_prior | v4_lite/nominal_train | v4_lite | 46 | 1.6172 | 4.0879 | 97.8% | 2.2% |
| v4lite__ablation_ablate_no_mass_prior_seed43 | ablation/ablate_no_mass_prior | v4_lite/nominal_train | iid_noisy_ic | 43 | 1.6385 | 4.1478 | 98.9% | 1.1% |
| iid__ablation_ablate_no_mass_prior_seed43 | ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | iid_noisy_ic | 43 | 1.6696 | 4.9135 | 97.8% | 2.2% |
| v4lite__ablation_ablate_no_mass_prior_seed43 | ablation/ablate_no_mass_prior | v4_lite/nominal_train | v4_lite | 43 | 1.8245 | 3.7290 | 98.9% | 1.1% |
| iid__ablation_ablate_no_mass_prior_seed45 | ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | iid_noisy_ic | 45 | 1.8488 | 5.2243 | 97.8% | 2.2% |
| clean__ablation_ablate_no_mass_prior_seed43 | ablation/ablate_no_mass_prior | clean | iid_noisy_ic | 43 | 1.8779 | 6.7264 | 98.9% | 1.1% |
| iid__ablation_ablate_no_mass_prior_seed45 | ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | v4_lite | 45 | 2.0006 | 4.9422 | 97.8% | 2.2% |
| iid__ablation_ablate_no_mass_prior_seed43 | ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | v4_lite | 43 | 2.0190 | 4.8092 | 97.8% | 2.2% |
| clean__ablation_ablate_no_mass_prior_seed43 | ablation/ablate_no_mass_prior | clean | v4_lite | 43 | 2.0658 | 6.3812 | 98.9% | 1.1% |
| v4lite__ablation_ablate_no_lift_seed46 | ablation/ablate_no_lift | v4_lite/nominal_train | iid_noisy_ic | 46 | 2.3225 | 5.4427 | 98.9% | 1.1% |
| v4lite__ablation_ablate_no_lift_seed46 | ablation/ablate_no_lift | v4_lite/nominal_train | v4_lite | 46 | 2.3983 | 5.6994 | 98.9% | 1.1% |
| clean__ablation_ablate_no_lift_seed43 | ablation/ablate_no_lift | clean | v4_lite | 43 | 44.3930 | 74.7825 | 100.0% | 0.0% |
| clean__ablation_ablate_no_lift_seed43 | ablation/ablate_no_lift | clean | iid_noisy_ic | 43 | 44.4952 | 74.6566 | 100.0% | 0.0% |

## Clean To nominal_eval Degradation @60s

| Model | Train | Eval | Pos/Clean Ratio | Completion Drop | Pos Degradation |
| --- | --- | --- | --- | --- | --- |
| main/phnode_full | clean | v4_lite | 1.309 | 0.0% | +30.9% |
| main/phnode_full | clean | iid_noisy_ic | 1.510 | 0.0% | +51.0% |
| ablation/ablate_no_lift | iid_noisy_ic/nominal_train | iid_noisy_ic | 1.218 | 0.0% | +21.8% |
| main/phnode_full | iid_noisy_ic/nominal_train | v4_lite | 1.179 | -0.2% | +17.9% |
| ablation/ablate_no_lift | iid_noisy_ic/nominal_train | v4_lite | 1.289 | 0.0% | +28.9% |
| main/phnode_full | iid_noisy_ic/nominal_train | iid_noisy_ic | 1.264 | 0.0% | +26.4% |
| main/phnode_full | v4_lite/nominal_train | iid_noisy_ic | 1.170 | -0.2% | +17.0% |
| main/phnode_full | v4_lite/nominal_train | v4_lite | 1.170 | -0.2% | +17.0% |
| ablation/ablate_no_lift | v4_lite/nominal_train | iid_noisy_ic | 1.159 | 0.0% | +15.9% |
| ablation/ablate_no_lift | v4_lite/nominal_train | v4_lite | 1.181 | 0.0% | +18.1% |
| ablation/ablate_no_mass_prior | v4_lite/nominal_train | iid_noisy_ic | 1.118 | 0.0% | +11.8% |
| ablation/ablate_no_mass_prior | v4_lite/nominal_train | v4_lite | 1.119 | -0.2% | +11.9% |
| ablation/ablate_no_mass_prior | clean | iid_noisy_ic | 1.112 | 0.0% | +11.2% |
| ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | iid_noisy_ic | 1.141 | 0.0% | +14.1% |
| ablation/ablate_no_mass_prior | clean | v4_lite | 1.140 | 0.0% | +14.0% |
| ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | v4_lite | 1.162 | 0.0% | +16.2% |
| ablation/ablate_no_lift | clean | v4_lite | 1.211 | 0.0% | +21.1% |
| ablation/ablate_no_lift | clean | iid_noisy_ic | 1.277 | 0.0% | +27.7% |

## Clean Replay Cost @60s

| Model | Train | Heldout Clean Ratio | Rollout Clean Ratio | Rollout Clean Degradation |
| --- | --- | --- | --- | --- |
| ablation/ablate_no_lift | iid_noisy_ic/nominal_train | 1.033 | 0.955 | -4.5% |
| main/phnode_full | iid_noisy_ic/nominal_train | 1.081 | 1.469 | +46.9% |
| main/phnode_full | v4_lite/nominal_train | 1.145 | 1.677 | +67.7% |
| ablation/ablate_no_lift | v4_lite/nominal_train | 1.156 | 1.135 | +13.5% |
| ablation/ablate_no_mass_prior | v4_lite/nominal_train | 1.009 | 1.009 | +0.9% |
| ablation/ablate_no_mass_prior | iid_noisy_ic/nominal_train | 1.009 | 1.032 | +3.2% |

## Notes

- `Train` uses `clean` or `protocol/profile` so Phase-1A can distinguish clean, iid noisy-IC, and `v4-lite` training runs.
- `Eval` distinguishes iid noisy-state and `v4-lite` noisy-state rollout rows when both are present for the same profile.
- Rollout sections use the selected primary eval profile for headline tables and keep `10s/30s/60s` in the horizon table.
- `Clean To ... Degradation` compares noisy eval against the same run's clean eval.
- `Clean Replay Cost` compares a noisy-trained run's clean eval against the matched clean-trained baseline with the same model and seed when available.
- Phase-1A is a protocol-sensitivity check; it does not by itself establish full family-level robustness against black-box baselines.


## 10. Optional diagnostics

`degraded_eval` 和 `heading_biased_eval` 仅作为后续诊断或 Phase-1B 条件扩展，不在本 notebook 的默认 clean run 中执行。